# From Words to Embeddings - Concepts

**Business problem:** turn email text into numbers, then decide which messages need a spam review. Later, reuse the same steps for job categories.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Distinguish counts, weighted counts and learned word vectors.
- Calculate a dot product and cosine similarity.
- Explain what static word lookup and averaging lose.

## Dataset and practical scope

The short messages and numerical word vectors below are constructed examples. The spam demonstration uses invented vectors, as on slide 11, and fits a real classifier to synthetic labels. It does not train word2vec or reproduce results on real emails.

All code is supplied. Run the cells from top to bottom.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. A lexical baseline

TF-IDF weights word identities. Different spellings remain different coordinates. Vocabulary and word weights are learned from training text only.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
texts = ['claim prize', 'claim reward', 'meeting team']
vectorizer = TfidfVectorizer()
counts = vectorizer.fit_transform(texts)
display(pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names_out()))
display(pd.DataFrame(cosine_similarity(counts), index=texts, columns=texts).round(3))

,claim,meeting,prize,reward,team
0,0.605349,0.000000,0.795961,0.000000,0.000000
1,0.605349,0.000000,0.000000,0.795961,0.000000
2,0.000000,0.707107,0.000000,0.000000,0.707107


,claim prize,claim reward,meeting team
claim prize,1.000,0.366,0.0
claim reward,0.366,1.000,0.0
meeting team,0.000,0.000,1.0


## 2. How word2vec learns

Skip-gram uses a centre word to predict nearby words. A learned input table supplies one word vector; a separate context table supplies prediction scores. Softmax turns scores into context-word probabilities. The loss uses an observed neighbouring word, not a spam or job-category label.

In [3]:
scores = np.array([2.0, 0.0, 1.0, -1.0, -2.0])
probabilities = np.exp(scores - scores.max())
probabilities /= probabilities.sum()
display(pd.DataFrame({'word': ['claim', 'prize', 'reward', 'meeting', 'team'],
                      'constructed score': scores, 'context probability': probabilities}))

,word,constructed score,context probability
0,claim,2.0,0.636409
1,prize,0.0,0.086129
2,reward,1.0,0.234122
3,meeting,-1.0,0.031685
4,team,-2.0,0.011656


## 3. Meaning as distance: use cosine carefully

A dot product multiplies matching entries and adds them. Cosine divides by the two vector lengths. It is not a probability, proof of synonymy or a causal explanation.

In [4]:
a, b = np.array([1.0, 2.0]), np.array([0.0, 1.0])
dot_product = a @ b
cosine = dot_product / (np.linalg.norm(a) * np.linalg.norm(b))
print('Dot product:', dot_product, '| Cosine:', round(cosine, 3))

Dot product: 2.0 | Cosine: 0.894


## 4. Average word vectors into a document

The following lookup table is invented. A mean of the same static word vectors ignores order.

In [5]:
word_vectors = {'bank': [1., 0.], 'paid': [0., 1.], 'supplier': [1., 1.]}
examples = ['bank paid supplier', 'supplier paid bank']
document_vectors = np.array([
    np.mean([word_vectors[word] for word in text.split()], axis=0) for text in examples])
display(pd.DataFrame(document_vectors, index=examples, columns=['value 1', 'value 2']))

,value 1,value 2
bank paid supplier,0.666667,0.666667
supplier paid bank,0.666667,0.666667


## Interpretation

Who pays whom in each example? Why can the static mean not recover that distinction? A contextual encoder can use positions before pooling, but does not guarantee correctness.

## 5. Back to the slides: which emails need a spam review?

**Worked example:** email text → mean word vector → logistic regression → estimated spam probability → threshold → review.

On slide 11, `claim = [0, 1]` and `prize = [1, 2]`, so `claim prize` becomes `[0.5, 1.5]`. We keep those exact values below. The other vectors and all message labels are invented for teaching, not learned word2vec output. A real word2vec model would learn its lookup table from training text, without spam labels; the classifier would then learn from labelled training emails.

These short phrases stand in for complete emails. Training, validation and test phrases are separate, but deliberately simple. No raw email data or model download is needed.

In [6]:
spam_word_vectors = {
    'claim': [0., 1.], 'prize': [1., 2.], 'reward': [1., 1.8],
    'win': [0.8, 1.8], 'free': [0.5, 1.5], 'offer': [0.5, 1.2],
    'meeting': [-1., -1.], 'team': [-1., -0.5],
    'project': [-0.8, -0.7], 'report': [-0.6, -1.]
}

def spam_email_mean(text):
    return np.mean([spam_word_vectors[word] for word in text.split()], axis=0)

spam_parts = {
    'train': pd.DataFrame([
        ('claim prize', 1), ('win reward', 1), ('free prize', 1),
        ('claim offer', 1), ('win free prize', 1), ('reward offer', 1),
        ('team meeting', 0), ('project report', 0), ('team project', 0),
        ('meeting report', 0), ('team meeting report', 0), ('project meeting', 0)
    ], columns=['text', 'spam']),
    'validation': pd.DataFrame([
        ('claim reward', 1), ('free offer', 1), ('win prize', 1),
        ('team report', 0), ('project team meeting', 0), ('report meeting project', 0)
    ], columns=['text', 'spam']),
    'test': pd.DataFrame([
        ('claim free reward', 1), ('win offer', 1), ('free reward', 1),
        ('meeting team project', 0), ('report project team', 0), ('team reward', 0)
    ], columns=['text', 'spam'])
}
spam_vectors = {name: np.array([spam_email_mean(text) for text in frame['text']])
                for name, frame in spam_parts.items()}
print('Slide 11: claim prize →', spam_email_mean('claim prize'))
display(pd.DataFrame({'messages': {name: len(frame) for name, frame in spam_parts.items()}}))

Slide 11: claim prize → [0.5 1.5]


,messages
train,12
validation,6
test,6


### Fit the classifier, not the word table

Start with an always-normal baseline: review nothing. Compare TF-IDF plus logistic regression with logistic regression on the two-value email means. TF-IDF learns its vocabulary and weights from training messages only. The invented word table stays fixed.

A spam probability comes from the fitted classifier, not from cosine similarity. With this tiny constructed dataset, treat it as a model estimate, not a trustworthy real-world risk.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score

spam_tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(random_state=SEED))
spam_tfidf.fit(spam_parts['train']['text'], spam_parts['train']['spam'])
spam_classifier = LogisticRegression(random_state=SEED)
spam_classifier.fit(spam_vectors['train'], spam_parts['train']['spam'])
spam_probabilities = {
    name: {
        'Always normal': np.zeros(len(spam_parts[name])),
        'TF-IDF': spam_tfidf.predict_proba(spam_parts[name]['text'])[:, 1],
        'Illustrative vector mean': spam_classifier.predict_proba(spam_vectors[name])[:, 1]
    }
    for name in ['validation', 'test']
}

### Choose a review threshold on validation messages

Suppose an unnecessary review of a normal message costs **5 units**, while missing spam costs **1 unit**. These are invented costs, not a recommendation for an actual email system. Try three thresholds on validation messages and select the lowest validation cost for each model; on a tie, keep the first threshold. Then evaluate once on the test messages.

**Precision** is the share of flagged messages that really are spam. **Recall** is the share of all spam messages that we flag. Neither alone describes the review workload or its cost.

In [8]:
def spam_review_cost(labels, predictions):
    normal_ok, false_alarms, missed_spam, spam_found = confusion_matrix(
        labels, predictions, labels=[0, 1]).ravel()
    return 5 * false_alarms + missed_spam

thresholds = [0.3, 0.5, 0.7]
spam_rows = []
selected_thresholds = {}
for name, probabilities in spam_probabilities['validation'].items():
    costs = [spam_review_cost(spam_parts['validation']['spam'], probabilities >= t)
             for t in thresholds]
    threshold = thresholds[int(np.argmin(costs))]
    selected_thresholds[name] = threshold
    predicted = spam_probabilities['test'][name] >= threshold
    actual = spam_parts['test']['spam']
    spam_rows.append({
        'model': name, 'threshold': threshold, 'validation cost': min(costs),
        'test precision': precision_score(actual, predicted, zero_division=0),
        'test recall': recall_score(actual, predicted, zero_division=0),
        'test reviews': int(predicted.sum()), 'test cost': spam_review_cost(actual, predicted)
    })
display(pd.DataFrame(spam_rows).set_index('model').round(3))
spam_test = spam_parts['test'].copy()
spam_test['estimated spam probability'] = spam_probabilities['test']['Illustrative vector mean']
spam_test['send to review'] = spam_test['estimated spam probability'] >= selected_thresholds['Illustrative vector mean']
display(spam_test.round(3))

,threshold,validation cost,test precision,test recall,test reviews,test cost
model,,,,,,
Always normal,0.3,3,0.00,0.0,0,3
TF-IDF,0.5,0,0.75,1.0,4,5
Illustrative vector mean,0.3,0,0.75,1.0,4,5


,text,spam,estimated spam probability,send to review
0,claim free reward,1,0.879,True
1,win offer,1,0.901,True
2,free reward,1,0.924,True
3,meeting team project,0,0.104,False
4,report project team,0,0.115,False
5,team reward,0,0.624,True


### Discuss: a reward can also be legitimate

- The test message `team reward` is labelled normal. Does the vector-mean classifier flag it? Inspect the row rather than assuming a dense representation wins.
- The always-normal baseline makes no false alarms, but misses every spam message. Here its precision is displayed as zero because it flags nothing.
- A lower threshold usually sends more messages to review. Would you keep the same costs if missing spam could cause a serious loss?

**Reading this run:** both fitted models find all three spam messages, but also flag `team reward`. That false alarm costs 5 units, more than the always-normal baseline's 3 missed-spam units. Better recall does not automatically mean a better business result. Do not retune the threshold on these test results; a revised policy needs fresh evaluation data.

The word table was designed to separate most of these phrases. This comparison cannot establish that word2vec is better than TF-IDF, or that its probabilities are reliable. Real emails include new words, changing language and ambiguous context; a phrase outside this small lookup table needs a different handling strategy. The toy function deliberately accepts only its listed words.

**Next practical step:** the [job-category exercise](text_embeddings_exercise.ipynb) uses the same representation → classifier → evaluation workflow, but actually learns word2vec from training text. Training and evaluating a real spam detector is an optional extension requiring an authorised labelled text corpus; the feature-only email CSV does not contain the words needed for this.

## Summary and business implications

Keep TF-IDF as a baseline. Word2vec learns related use, not guaranteed meaning. Train a separate classifier and test it on new documents.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric. The spam lookup table and costs are invented, and six test messages cannot support a deployment decision.

## Further exercises

- Train word2vec and choose your own words to visualize in the [word-map exercise](word_relationships_exercise.ipynb), inspired by page 17 of *Slides_MSC_Data_Science_Week_08_EN.pdf*. Compare the picture with cosine scores calculated using all 100 numbers per word.
- Explain why shared contexts can also connect antonyms.
- What should happen when a message contains no known words?
- Complete the job-category exercise; its small synthetic dataset is not a business benchmark.

## References

**Required reading:** Stanford CS224N, Spring 2024, word-vector lecture, supplied excerpt.
**Instructor reference:** Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space*.
https://arxiv.org/abs/1301.3781
**Official documentation:** https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.